#### Steps taken:
1. Read silver `drivers` table
2. Read gold `ref_nationality_region` table
3. Join the data from `drivers` with `ref_nationality_region` using `nationality`
4. Select the required columns
 - drivers.driver_id
 - drivers.driver_name
 - drivers.date_of_birth
 - drivers.nationality
 - ref_nationality_region.region
5. Write the transformed data to gold `dim_drivers` table

In [0]:
%run ../environment_config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog}.{gold_schema}.dim_drivers"

In [0]:
drivers_df = spark.read.table(f"{catalog}.{silver_schema}.drivers")
ref_nationality_region = spark.read.table(f"{catalog}.{gold_schema}.ref_nationality_region")

In [0]:
dim_drivers_df = (
                 drivers_df.alias("d").join(
                                 ref_nationality_region.alias("n"),
                                 F.col("d.nationality") == F.col("n.nationality"),
                                 "left"
                                          )
                                       .select(
                                           drivers_df.driver_id,
                                           drivers_df.driver_name,
                                           drivers_df.date_of_birth,
                                           drivers_df.nationality,
                                           ref_nationality_region.region.alias("nationality_region")
                                       )
               )

In [0]:
display(dim_drivers_df)

In [0]:
(
    dim_drivers_df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(target_table)
)

In [0]:
display(spark.table("formula1.gold.dim_drivers"))